# The Data Doctor: A Complete Machine Learning Project
Predicting which customers will leave a phone company, from raw data to a money-saving plan: cleaning, exploring, comparing models, choosing a threshold by cost, and explaining the result. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. Meet the patient
5,000 customers of a phone and internet company (made up for this video). About one in four left last month (**churn**). Our job: find who is likely to leave next, so the company can act in time.

### Load the customers
- `df.shape`: 5,000 rows (customers), 12 columns
- `head(3)[[...]]`: five of the columns, for the first three customers

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "capstone-data-doctor/phone_customers.csv")
df = pd.read_csv(url)
print(df.shape)
print(df.head(3)[["tenure", "contract", "internet",
                "monthly_charges", "churn"]])

### Column types
- `dtypes`: the type of each column
- `str (or object)`: text; total_charges should be a number, so something is wrong

In [ ]:
print(df.dtypes)

## 2. Run some tests: cleaning
total_charges is stored as text because 51 brand-new customers (tenure 0) have a blank. Convert it to numbers; a blank becomes NaN; a new customer has paid nothing yet, so fill with 0.

### Fix total charges
- `errors="coerce"`: anything that isn't a number becomes NaN (missing)
- `unique()`: the missing ones all have tenure 0
- `fillna(0)`: new customers have paid nothing yet
- `left`: 1 if the customer left, 0 if they stayed

In [ ]:
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")
print("missing:", df["total_charges"].isna().sum())
print(df.loc[df["total_charges"].isna(), "tenure"].unique())
df["total_charges"] = df["total_charges"].fillna(0)
df["left"] = (df["churn"] == "yes").astype(int)
print("churn rate:", round(df["left"].mean(), 3))

## 3. The examination: exploring
Before any model, look. Which kinds of customer leave most?

### Churn rate by group
- `groupby(col)["left"].mean()`: the share who left, in each group
- `to_dict()`: print the groups on one line
- `the output`: month-to-month, fiber and electronic check stand out

In [ ]:
for col in ["contract", "internet", "tech_support", "payment"]:
    rates = df.groupby(col)["left"].mean().round(2).sort_values()
    print(col, rates.to_dict())

### Two charts
- `plot.bar`: a bar for each contract type
- `a2.hist([...])`: tenure of stayers and leavers, side by side
- `the picture`: month-to-month customers leave most; most leavers are new customers

In [ ]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
df.groupby("contract")["left"].mean().plot.bar(ax=a1, rot=0)
a1.set_ylabel("share who left"); a1.set_title("by contract")
a2.hist([df.loc[df.left == 0, "tenure"], df.loc[df.left == 1, "tenure"]],
         bins=24, label=["stayed", "left"])
a2.set_xlabel("tenure (months)"); a2.legend(); a2.set_title("by tenure")
plt.show()

## 4. Prepare the data
Inputs X, target y. Hold back 20% as a test set. Text columns get **one-hot encoding**; number columns get **scaled**. A pipeline keeps the steps together.

### Split and set up the preparation
- `drop(columns=...)`: the ID and the answers are not inputs
- `stratify=y`: the same churn rate in both parts
- `OneHotEncoder`: one 0/1 column per category
- `StandardScaler`: numbers centred on 0 with spread 1
- `ColumnTransformer`: apply each step to the right columns

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=["customer_id", "churn", "left"])
y = df["left"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=0, stratify=y)
text_cols = ["partner", "contract", "internet", "tech_support",
             "paperless_billing", "payment"]
num_cols = ["senior", "tenure", "monthly_charges", "total_charges"]
prep = ColumnTransformer([("text", OneHotEncoder(), text_cols),
                          ("num", StandardScaler(), num_cols)])
print(len(X_tr), "to train,", len(X_te), "to test")

## 5. A baseline, and the accuracy trap
A 'model' that always says **nobody leaves** is right about 74% of the time, and catches no one. Accuracy alone can fool you.

### The do-nothing doctor
- `most_frequent`: always predict the most common answer: stays
- `leavers caught`: how many real leavers it flags: none

In [ ]:
from sklearn.dummy import DummyClassifier

lazy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print("accuracy:", round(lazy.score(X_te, y_te), 3))
print("leavers caught:", (lazy.predict(X_te)[y_te == 1] == 1).sum(), "of", y_te.sum())

## 6. Try three treatments
Compare logistic regression, a random forest and gradient boosting with 5-fold **cross-validation**, scored by **ROC AUC** (the chance a random leaver gets a higher risk score than a random stayer: 0.5 is guessing, 1 is perfect).

### Cross-validate three models
- `make_pipeline(prep, m)`: preparation + model, refitted inside every fold: no leaks
- `cv=5`: five folds
- `the output`: all close; the simplest model is slightly best

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {"logistic regression": LogisticRegression(max_iter=1000),
          "random forest": RandomForestClassifier(300, min_samples_leaf=5,
                                                  random_state=0),
          "gradient boosting": GradientBoostingClassifier(random_state=0)}
for name, m in models.items():
    auc = cross_val_score(make_pipeline(prep, m), X_tr, y_tr, cv=5,
                          scoring="roc_auc")
    print(f"{name:<20} ROC AUC {auc.mean():.3f}")

## 7. Check the treatment works
Fit the winner on all the training data and test it once on the held-back customers. Look at the **confusion matrix**, **precision** (of those we flag, how many really leave) and **recall** (of those who leave, how many we flag).

### Test the winner
- `predict_proba(...)[:, 1]`: each customer's risk: the probability of leaving
- `risk > 0.5`: flag customers above 50% risk
- `confusion_matrix`: rows: really stayed / left; columns: not flagged / flagged

In [ ]:
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                             precision_score, recall_score)

model = make_pipeline(prep, LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
risk = model.predict_proba(X_te)[:, 1]
flag = risk > 0.5
print("test ROC AUC:", round(roc_auc_score(y_te, risk), 3))
print(confusion_matrix(y_te, flag))
print("precision:", round(precision_score(y_te, flag), 3),
      " recall:", round(recall_score(y_te, flag), 3))

## 8. Explain the diagnosis
Logistic regression gives one **coefficient** per input: positive pushes risk up, negative pushes it down.

### What drives the risk?
- `get_feature_names_out`: the column names after one-hot encoding and scaling
- `coef_`: the learned coefficients
- `the output`: long tenure and two-year contracts lower risk; month-to-month and fiber raise it

In [ ]:
names = model[0].get_feature_names_out()
coefs = pd.Series(model[1].coef_[0], index=names).sort_values()
print("lowers risk:\n", coefs.head(4).round(2))
print("raises risk:\n", coefs.tail(4).round(2))

## 9. The prescription: who to call
A leaver costs about **$500**. A retention offer costs **$50** and keeps **40%** of real leavers. So each leaver we call saves 0.4 x 500 - 50 = **$150**; each false alarm costs **$50**. Find the threshold that saves the most.

### Savings for every threshold
- `caught * 0.4 * 500`: leavers kept, times what each would have cost
- `called.sum() * 50`: every call costs an offer
- `savings(-1)`: a threshold below 0: call everyone
- `the picture`: a hill: too few calls miss leavers, too many waste offers

In [ ]:
import numpy as np

def savings(t):
    called = risk > t
    caught = (called & (y_te == 1)).sum()
    return caught * 0.4 * 500 - called.sum() * 50

thresholds = np.arange(0.05, 0.96, 0.05)
money = [savings(t) for t in thresholds]
best = thresholds[np.argmax(money)]
for t in [0.2, 0.5]:
    print(f"threshold {t}: save ${savings(t):,.0f}")
print(f"call everyone: ${savings(-1):,.0f}   best threshold: {best:.2f}")
plt.figure(figsize=(10, 4))
plt.plot(thresholds, money, marker="o")
plt.axvline(best, color="gray", ls="--")
plt.xlabel("threshold"); plt.ylabel("money saved ($)"); plt.show()

## 10. Follow-up care
Save the model, load it back, and produce this month's call list. In real life, check its performance every month: customers change.

### Save, load and score
- `joblib.dump`: save the whole pipeline to a file
- `joblib.load`: load it back, ready to use
- `new`: a made-up new customer

In [ ]:
import joblib

joblib.dump(model, "churn_model.joblib")
loaded = joblib.load("churn_model.joblib")
new = pd.DataFrame([{"senior": 0, "partner": "no", "tenure": 2,
                     "contract": "month-to-month", "internet": "fiber",
                     "tech_support": "no", "paperless_billing": "yes",
                     "payment": "electronic check", "monthly_charges": 80.0,
                     "total_charges": 160.0}])
print("new customer risk:", round(loaded.predict_proba(new)[0, 1], 2))

### This month's call list
- `df["left"] == 0`: only customers who are still here
- `risk > 0.2`: our money-saving threshold
- `sort_values("risk")`: the riskiest customers first

In [ ]:
df["risk"] = loaded.predict_proba(X)[:, 1]
call_list = df[(df["left"] == 0) & (df["risk"] > 0.2)]
print(len(call_list), "current customers to call")
print(call_list.sort_values("risk", ascending=False)
      [["customer_id", "tenure", "contract", "risk"]].head(5).round(2))

## Try it yourself
1. Change the retention offer cost from 50 to 100 dollars. Which threshold is best now?
2. Add class_weight="balanced" to the logistic regression. How do precision and recall change?
3. Make up a new customer with a two-year contract and score them.